# Part B: Stretch Problem - Approximate Nearest Neighbors with FAISS

## 1. Install FAISS

In [6]:
# This cell installs the FAISS library, which is a library for efficient similarity search and clustering of dense vectors.
import sys

# The '!{sys.executable} -m pip install faiss-cpu' command uses pip to install the CPU-only version of FAISS.
# `sys.executable` ensures that pip is run using the same Python interpreter that is currently running the notebook.
!{sys.executable} -m pip install faiss-cpu

# These are the necessary libraries we'll be using throughout this problem:
# numpy for numerical operations, especially with arrays.
import numpy as np
# load_digits from sklearn.datasets to get a sample digit dataset.
# train_test_split from sklearn.model_selection to divide data into training and testing sets.
# NearestNeighbors from sklearn.neighbors for Scikit-learn's KNN implementation.
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
# faiss is the library we just installed, used for efficient similarity search.
import faiss
# time for measuring the execution speed of different KNN implementations.
import time
# pandas for data manipulation and analysis, though not heavily used in this specific problem.
import pandas as pd

## 2. Load and Prepare the Digits Dataset

In [7]:
# The load_digits() function fetches the optical recognition of handwritten digits dataset.
digits = load_digits()
# X contains the 8x8 pixel data for each digit (64 features).
X = digits.data
# y contains the actual digit (0-9) that each image represents.
y = digits.target

# Print the shape of the full dataset (number of samples, number of features).
print(f"Shape of the dataset: {X.shape}")

# We split the data into a reference set (for building the index) and a query set (for searching).
# train_test_split divides X and y, with 80% for reference (training) and 20% for query (testing).
# random_state ensures reproducibility of the split.
X_ref, X_query, y_ref, y_query = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes of the reference and query sets.
print(f"Shape of reference set: {X_ref.shape}")
print(f"Shape of query set: {X_query.shape}")

# d stores the dimensionality of our data (number of features), which is 64.
d = X.shape[1]

Shape of the dataset: (1797, 64)
Shape of reference set: (1437, 64)
Shape of query set: (360, 64)


## 3. Implement KNN search using FAISS

In [8]:
# Normalizing the data is crucial for distance-based algorithms like FAISS and KNN.
# The pixel values in the digits dataset range from 0 to 16. Dividing by 16.0 scales them to 0-1.
# We also cast them to float32, which is often preferred by FAISS for performance.
X_ref_normalized = X_ref.astype('float32') / 16.0
X_query_normalized = X_query.astype('float32') / 16.0

# Build a FAISS index. IndexFlatL2 is a simple brute-force index that calculates L2 (Euclidean) distances.
# It's an exact search method, meaning it finds the true nearest neighbors.
# The 'd' parameter is the dimensionality of the vectors we are indexing (64 in our case).
index = faiss.IndexFlatL2(d)

# Check if the index needs to be trained. For IndexFlatL2, it typically doesn't, so this will be True.
print(f"Is the index trained? {index.is_trained}")

# Add the reference vectors to the FAISS index.
# The index now stores these vectors and can perform searches on them.
index.add(X_ref_normalized)
print(f"Number of vectors in the index: {index.ntotal}")

Is the index trained? True
Number of vectors in the index: 1437


## 4. Perform KNN Search and Compare Speed

In [10]:
# Define k, the number of nearest neighbors we want to retrieve for each query.
k = 5

# --- FAISS KNN Search ---
print("Performing FAISS KNN search...")
# Record the start time for FAISS search.
start_time_faiss = time.time()
# index.search(query_vectors, k) performs the search.
# D_faiss will contain the distances to the k nearest neighbors.
# I_faiss will contain the indices of the k nearest neighbors in the reference set.
D_faiss, I_faiss = index.search(X_query_normalized, k)
# Record the end time for FAISS search.
end_time_faiss = time.time()
# Calculate the total time taken by FAISS.
faiss_time = end_time_faiss - start_time_faiss
print(f"FAISS search time: {faiss_time:.4f} seconds")

# --- Scikit-learn KNN Search ---
print("Performing Scikit-learn KNN search...")
# For a fair comparison, we use Scikit-learn's NearestNeighbors with 'brute' algorithm and 'euclidean' metric.
# This ensures it also performs an exact brute-force search, similar to FAISS IndexFlatL2.
nbrs = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean').fit(X_ref_normalized)

# Record the start time for Scikit-learn search.
start_time_sklearn = time.time()
# nbrs.kneighbors(query_vectors) performs the search.
# D_sklearn will contain distances, and I_sklearn will contain indices.
D_sklearn, I_sklearn = nbrs.kneighbors(X_query_normalized)
# Record the end time for Scikit-learn search.
end_time_sklearn = time.time()
# Calculate the total time taken by Scikit-learn.
sklearn_time = end_time_sklearn - start_time_sklearn
print(f"Scikit-learn search time: {sklearn_time:.4f} seconds")

print("\n--- Verification (Optional) ---")
# We verify the results. Due to potential floating-point precision differences,
# distances might vary slightly, but the indices of the nearest neighbors should be largely the same.
# Let's inspect the first query's nearest neighbors found by both methods.
print(f"First query (FAISS indices): {I_faiss[0]}")
print(f"First query (Sklearn indices): {I_sklearn[0]}")

# Calculate the percentage of query sets where the top-k neighbor indices are identical.
matches = np.mean([np.array_equal(I_faiss[i], I_sklearn[i]) for i in range(len(I_faiss))])
print(f"Percentage of identical top-k neighbor sets: {matches:.2%}")

# --- Documentation of Findings ---
print("\n## 5. Document Findings")
print(f"- FAISS (IndexFlatL2) is generally faster for exact nearest neighbor search, especially with larger datasets and query sets.")
print(f"- FAISS search took {faiss_time:.4f} seconds, while Scikit-learn's brute-force KNN took {sklearn_time:.4f} seconds.")


Performing FAISS KNN search...
FAISS search time: 0.0158 seconds
Performing Scikit-learn KNN search...
Scikit-learn search time: 0.0348 seconds

--- Verification (Optional) ---
First query (FAISS indices): [ 713  428  805 1161 1253]
First query (Sklearn indices): [ 713  428  805 1161 1253]
Percentage of identical top-k neighbor sets: 96.67%

## 5. Document Findings
- FAISS (IndexFlatL2) is generally faster for exact nearest neighbor search, especially with larger datasets and query sets.
- FAISS search took 0.0158 seconds, while Scikit-learn's brute-force KNN took 0.0348 seconds.


## 5. Document Findings
- FAISS (IndexFlatL2) is generally faster for exact nearest neighbor search, especially with larger datasets and query sets.
- FAISS search took 0.0129 seconds, while Scikit-learn's brute-force KNN took 0.0190 seconds.
- In this specific case, FAISS was 1.48 times faster than Scikit-learn.
- The IndexFlatL2 in FAISS performs an exhaustive search, guaranteeing accuracy similar to Scikit-learn's brute-force approach.
- For approximate nearest neighbors (ANN), FAISS offers various index types (e.g., LSH, IVFPQ) that trade off some accuracy for significant speed improvements on much larger datasets.